📚 PROYECTO INTEGRADOR: Pipeline ETLT de Clima a Data Lake

Este notebook consolida el código fuente, la configuración y los pasos de ejecución del Pipeline de Ingeniería de Datos basado en PySpark y Apache Airflow, utilizando un Data Lake en AWS S3.


1. Configuración de Infraestructura y Dependencias


Esta sección describe la infraestructura base requerida (Docker para Airflow) y las librerías necesarias.

A. Dependencias de Python (requirements.txt)

El proyecto requiere estas librerías para su desarrollo y ejecución de pruebas:


In [4]:
### B. Inicialización de Airflow con Docker

La orquestación se realiza en un entorno Docker-Compose. Asumiendo que el `docker-compose.yml` está en el directorio `infra/`:

| Celda de Código: Comando de Inicialización |
| :--- |
| ```bash
# 1. Navegar a la carpeta de infraestructura
cd infra/

# 2. Levantar los servicios de Airflow (Webserver, Scheduler, Postgres)
docker compose up -d
``` |

---

## 2. Avance #2: Script de Transformación PySpark (`scripts/transform_job.py`)

Este script se conecta al Data Lake en S3 (capa RAW), aplica la lógica ETLT, y escribe los resultados en Parquet (capa PROCESSED). Contiene las correcciones cruciales para el manejo de la conexión S3A.

| Celda de Código: `scripts/transform_job.py` |
| :--- |
| ```python
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_timestamp, round, explode, to_date

# 1. Iniciar Spark Session y Configurar AWS
# Se inyectan las correcciones para los errores de formato de tiempo de S3A.
spark = SparkSession.builder.appName("PI_Transformacion_Clima") \
    \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "com.amazonaws.auth.EnvironmentVariableCredentialsProvider") \
    \
    .config("spark.hadoop.fs.s3a.threadpool.core.keepaliveTime", "60000") \
    .config("spark.hadoop.fs.s3a.connection.establish.timeout", "5000") \
    .config("spark.hadoop.fs.s3a.connection.timeout", "60000") \
    .config("spark.hadoop.fs.s3a.threads.keepalivetime", "60000") \
    \
    .config("spark.hadoop.fs.s3a.multipart.purge.age", "86400000") \
    \
    .getOrCreate()

# 2. Rutas del Data Lake 
S3_BUCKET_NAME = "pi-data-lake-alejandro"  
S3_BASE_PATH = f"s3a://{S3_BUCKET_NAME}" 
RAW_PATH = f"{S3_BASE_PATH}/raw/weather_data/" 
PROCESSED_PATH = f"{S3_BASE_PATH}/processed/clima_historico/"

# 3. Leer Datos Crudos y Aplicar EXPLODE
try:
    print(f"Leyendo archivos JSON anidados desde: {RAW_PATH}")
    df_raw = spark.read.json(RAW_PATH, multiLine=True)
    
    # Aplanar el array 'weather' para acceder a sus campos
    df_exploded = df_raw.withColumn("weather_data", explode(col("weather")))
    
except Exception as e:
    print(f"Error CRÍTICO al leer JSON de S3: {e}")
    spark.stop()
    exit()

# 4. Transformación de Datos (Selección, Tipado y Normalización)
df_transformado = df_exploded.select(
    # Extracción de campos anidados (Flattening)
    col("dt_iso").cast("timestamp").alias("fecha_hora_utc"),
    to_date(col("dt_iso")).alias("fecha_analisis"), 
    col("city_name").alias("ciudad"),
    col("lat").alias("latitud"),
    col("lon").alias("longitud"),
    # Datos de MAIN
    round(col("main.temp").cast("double"), 2).alias("temperatura_celsius"),
    col("main.humidity").alias("humedad_porcentaje"),
    col("main.pressure").alias("presion_hpa"),
    # Datos de WEATHER (del Array aplanado)
    col("weather_data.main").alias("condicion_clima_principal"),
    col("weather_data.description").alias("descripcion_clima"),
    # Columna de auditoría
    current_timestamp().alias("fecha_procesamiento_pipeline")
)

# 5. Escribir Resultados en la Capa PROCESSED (Parquet)
print(f"Escribiendo datos procesados en: {PROCESSED_PATH}")
df_transformado.write.mode("overwrite").partitionBy("ciudad", "fecha_analisis") \
                 .parquet(PROCESSED_PATH)

print("--- ¡Transformación PySpark completada! Datos listos en formato Parquet. ---")
spark.stop()
``` |

---

## 3. Avance #3 y #4: DAG de Orquestación Airflow (`dags/etl_weather_dag.py`)

Este DAG define el flujo para ejecutar el script PySpark. Se ha corregido el error de `schedule_interval` a `schedule` para ser compatible con versiones recientes de Airflow.

| Celda de Código: `dags/etl_weather_dag.py` |
| :--- |
| ```python
from airflow.operators.bash import BashOperator
from airflow.models.dag import DAG
import pendulum
import os

# La ruta donde el contenedor de Airflow encontrará el script.
# (Asumimos que el DAG y el script se copian a la carpeta /opt/airflow/dags/)
TRANSFORM_SCRIPT_PATH = "/opt/airflow/dags/transform_job.py" 

with DAG(
    dag_id='etl_json_weather_to_parquet',
    start_date=pendulum.datetime(2025, 9, 29, tz="UTC"),
    # CORRECCIÓN: Usar 'schedule' en lugar de 'schedule_interval'
    schedule='@daily',
    catchup=False,
    tags=['DataLake', 'PySpark', 'JSON'],
    default_args={
        'owner': 'alejandro_arango',
        'retries': 1,
    }
) as dag:
    
    # Tarea que invoca el binario spark-submit, descargando el conector hadoop-aws
    pyspark_transform_task = BashOperator(
        task_id='ejecutar_transformacion_json_a_parquet',
        bash_command=f"""
        # Se asume que spark-submit está accesible en el PATH del contenedor
        spark-submit --packages org.apache.hadoop:hadoop-aws:3.3.1 {TRANSFORM_SCRIPT_PATH}
        """,
    )
    
    # Definición del flujo: solo una tarea
    pyspark_transform_task
``` |

---

## 4. Pasos de Ejecución y Verificación (Avance #4 Final)

Una vez que los servicios de Airflow están levantados, la ejecución se realiza a través de la UI.

### A. Despliegue de Archivos

Antes de ejecutar, asegúrese de que el archivo `etl_weather_dag.py` (corregido) y el `transform_job.py` estén en la carpeta local que su Docker está leyendo como volumen de DAGs.

### B. Ejecución en la Interfaz de Airflow

1.  **Acceso:** Abra su navegador y navegue a `http://localhost:8080`.
2.  **Activación:** Busque el DAG **`etl_json_weather_to_parquet`** y active el interruptor.
3.  **Disparo:** Haga clic en **"Trigger DAG"** (Disparar) para iniciar la ejecución.

### C. Verificación de Éxito

| Celda de Código: Verificación de Salida (Logs Esperados) |
| :--- |
| **Paso:** Revisar los logs de la tarea `ejecutar_transformacion_json_a_parquet` |
| ```text
[2025-09-29 20:00:00,000] {bash.py:175} INFO - ... (Salida de Spark y logs de conexión)
[2025-09-29 20:00:05,000] INFO - Leyendo archivos JSON anidados desde: s3a://pi-data-lake-alejandro/raw/weather_data/
[2025-09-29 20:00:10,000] INFO - Escribiendo datos procesados en: s3a://pi-data-lake-alejandro/processed/clima_historico/
[2025-09-29 20:00:15,000] INFO - --- ¡Transformación PySpark completada! Datos listos en formato Parquet. ---
``` |

SyntaxError: leading zeros in decimal integer literals are not permitted; use an 0o prefix for octal integers (2446610006.py, line 156)